# Pass-rush aggregates from `player_play` (game & season)

Manual notebook for Kevin Zhang (BDB 2027 / Idea A).

**Outputs (agent-blocked):**
- `DerivedData/player_play_passrush_by_game_player.csv` — one row per `(gameId, nflId)`
- `DerivedData/player_play_passrush_by_season_player.csv` — one row per `(season, nflId)`

Do **not** paste raw competition CSV rows into AI chats. Print column names / shape / aggregate summaries only.

## Concept → schema mapping (verify locally)

| Concept (Kevin) | Expected competition column(s) | Notes |
|---|---|---|
| lined-up role filter | `lined_up_position` (aliases: `linedUpPosition`) | Keep `EDGE`, `OLB`, `INTERIOR_LINE` only. Treat string `"NA"` as a real label (pre-snap penalty plays) — do not coerce it with default NA parsing alone. |
| Pass play | Prefer `hadDropback == 1` (BDB 2025 lineage on `player_play`). Fallback: `isDropback` / `passPlay` / non-null pass-rush timing on the row. | Confirm with `print(list(df.columns))` before trusting. |
| `time_to_pressure` | `timeToPressureAsPassRusher` | Continuous; **mean over non-NaN**. Also store event indicator + count + optional JSON list of non-null times. |
| `time_to_qb_hurry` | Prefer `timeToHurryAsPassRusher` / `timeToQbHurryAsPassRusher` / `timeToQuarterbackHurry`; else derive event from `quarterbackHit` if no hurry-time column exists | Times: mean non-NaN. Events: rate + count. |
| `sack` | Binary from `sackYardsAsDefense` / `halfSackYardsAsDefense` (nonzero ⇒ sack credit) or explicit `sack`/`gotSack` if present | Rate (mean) + count (sum). |
| `player_get_off` | `getOffTimeAsPassRusher` | Mean over non-NaN get-off times on valid rushes. |
| Pressure event | `causedPressure` | Rate + count (pressure is the primary disruption DV). |
| `blitzing` | `blitzing` / `isBlitzing` / `didBlitz` (optional) | Binary rate if present. |
| `unblocked_pressure` | `unblockedPressure` / `causedUnblockedPressure` (optional; was play-level in BDB 2025 `plays`) | Binary rate if present. |
| `quick_pressure` | `quickPressure` / `causedQuickPressure` (optional) | Binary rate if present. |

Aggregation rules: **means/rates** for times and binary indicators; **sums/counts** for event totals; optional JSON-serialized lists of non-null times per group.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Paths — edit if your local layout differs
# ---------------------------------------------------------------------------
REPO_ROOT = Path("../..").resolve()
DATA_DIR = REPO_ROOT / "Data"          # competition CSVs (agent-ignored)
DERIVED_DIR = REPO_ROOT / "DerivedData"  # outputs (agent-ignored)

PLAYER_PLAY = DATA_DIR / "player_play.csv"
GAMES = DATA_DIR / "games.csv"
PLAYERS = DATA_DIR / "players.csv"

OUT_GAME = DERIVED_DIR / "player_play_passrush_by_game_player.csv"
OUT_SEASON = DERIVED_DIR / "player_play_passrush_by_season_player.csv"

LINED_UP_KEEP = {"EDGE", "OLB", "INTERIOR_LINE"}
STORE_TIME_LISTS = True  # set False to skip JSON time-list columns

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR exists:", DATA_DIR.is_dir())
for p in (PLAYER_PLAY, GAMES, PLAYERS):
    print(f"  {p.name}: {'OK' if p.is_file() else 'MISSING'} -> {p}")


In [ ]:
def _norm(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def resolve_column(columns, *candidates, required: bool = False) -> str | None:
    """Match a column by exact name, casefold, or alphanumeric-normalized form."""
    cols = list(columns)
    by_cf = {c.casefold(): c for c in cols}
    by_norm = {_norm(c): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
        if cand.casefold() in by_cf:
            return by_cf[cand.casefold()]
        if _norm(cand) in by_norm:
            return by_norm[_norm(cand)]
    if required:
        raise KeyError(f"Required column not found. Tried: {candidates}. Have: {cols}")
    return None


def headers_only(path: Path) -> list[str]:
    return list(pd.read_csv(path, nrows=0).columns)


# Header-only verification (no row dump)
for label, path in [("player_play", PLAYER_PLAY), ("games", GAMES), ("players", PLAYERS)]:
    if not path.is_file():
        print(f"[skip] {label}: file missing")
        continue
    cols = headers_only(path)
    print(f"{label}: {len(cols)} columns")
    print(cols)


In [ ]:
# Resolve schema from headers (still no row contents printed)
assert PLAYER_PLAY.is_file(), f"Missing {PLAYER_PLAY} — set DATA_DIR at top of notebook"
pp_cols = headers_only(PLAYER_PLAY)
games_cols = headers_only(GAMES) if GAMES.is_file() else []
player_cols = headers_only(PLAYERS) if PLAYERS.is_file() else []

COL = {
    "gameId": resolve_column(pp_cols, "gameId", "game_id", required=True),
    "playId": resolve_column(pp_cols, "playId", "play_id", required=True),
    "nflId": resolve_column(pp_cols, "nflId", "nfl_id", required=True),
    "teamAbbr": resolve_column(pp_cols, "teamAbbr", "team_abbr", "club"),
    "lined_up": resolve_column(
        pp_cols, "lined_up_position", "linedUpPosition", "linedUpPos", required=True
    ),
    "hadDropback": resolve_column(pp_cols, "hadDropback", "isDropback", "passPlay", "isPassPlay"),
    "wasInitialPassRusher": resolve_column(pp_cols, "wasInitialPassRusher"),
    "time_to_pressure": resolve_column(pp_cols, "timeToPressureAsPassRusher", "time_to_pressure"),
    "time_to_qb_hurry": resolve_column(
        pp_cols,
        "timeToHurryAsPassRusher",
        "timeToQbHurryAsPassRusher",
        "timeToQuarterbackHurry",
        "timeToHurry",
        "time_to_qb_hurry",
    ),
    "get_off": resolve_column(pp_cols, "getOffTimeAsPassRusher", "player_get_off", "getOffTime"),
    "causedPressure": resolve_column(pp_cols, "causedPressure", "pressure"),
    "qb_hit": resolve_column(pp_cols, "quarterbackHit", "qbHit", "qb_hit"),
    "sack_yards": resolve_column(pp_cols, "sackYardsAsDefense"),
    "half_sack_yards": resolve_column(pp_cols, "halfSackYardsAsDefense"),
    "sack_flag": resolve_column(pp_cols, "sack", "gotSack", "wasSackedAsDefense"),
    "blitzing": resolve_column(pp_cols, "blitzing", "isBlitzing", "didBlitz", "blitz"),
    "unblocked_pressure": resolve_column(
        pp_cols, "unblockedPressure", "causedUnblockedPressure", "unblocked_pressure"
    ),
    "quick_pressure": resolve_column(
        pp_cols, "quickPressure", "causedQuickPressure", "quick_pressure"
    ),
}

GAMES_SEASON = resolve_column(game_cols, "season") if game_cols else None
GAMES_WEEK = resolve_column(game_cols, "week") if game_cols else None
GAMES_TYPE = resolve_column(game_cols, "seasonType", "season_type", "gameType", "game_type") if game_cols else None
PLAYERS_NAME = resolve_column(player_cols, "displayName", "playerName", "name") if player_cols else None
PLAYERS_POS = resolve_column(player_cols, "position") if player_cols else None

print("Resolved player_play mapping:")
for k, v in COL.items():
    print(f"  {k:24s} -> {v}")
print("games.season ->", GAMES_SEASON, "| week ->", GAMES_WEEK, "| type ->", GAMES_TYPE)
print("players name/pos ->", PLAYERS_NAME, PLAYERS_POS)

missing_core = [k for k in ("time_to_pressure", "get_off", "causedPressure") if COL[k] is None]
if missing_core:
    print("WARNING: core pass-rush columns missing:", missing_core)
if COL["hadDropback"] is None:
    print("WARNING: no explicit pass/dropback flag; will use pass-rush field presence as proxy.")


In [ ]:
# Load only needed columns (still: do not display raw row contents)
usecols = [c for c in COL.values() if c is not None]
# keep order unique
usecols = list(dict.fromkeys(usecols))

# Read lined_up as string so literal "NA" is preserved where possible
dtype = {COL["lined_up"]: "string"}
pp = pd.read_csv(PLAYER_PLAY, usecols=usecols, dtype=dtype, low_memory=False)
print("player_play loaded:", pp.shape, "| columns:", list(pp.columns))

games = None
if GAMES.is_file() and GAMES_SEASON is not None:
    g_usecols = [resolve_column(game_cols, "gameId", "game_id", required=True), GAMES_SEASON]
    if GAMES_WEEK:
        g_usecols.append(GAMES_WEEK)
    if GAMES_TYPE:
        g_usecols.append(GAMES_TYPE)
    games = pd.read_csv(GAMES, usecols=g_usecols, low_memory=False)
    print("games loaded:", games.shape)
else:
    print("WARNING: games.csv missing/unusable — season aggregates need a season column.")

players = None
if PLAYERS.is_file() and PLAYERS_NAME is not None:
    p_usecols = [resolve_column(player_cols, "nflId", "nfl_id", required=True), PLAYERS_NAME]
    if PLAYERS_POS:
        p_usecols.append(PLAYERS_POS)
    players = pd.read_csv(PLAYERS, usecols=p_usecols, low_memory=False)
    print("players loaded:", players.shape)


In [ ]:
def to_numeric_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.astype(float)
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    # bool-like strings / mixed
    mapped = s.replace({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0})
    return pd.to_numeric(mapped, errors="coerce")


def binary_event(s: pd.Series) -> pd.Series:
    x = to_numeric_series(s)
    # treat nonzero (incl. True/1) as event; NaN stays NaN then filled 0 for rate denominators later
    out = (x.fillna(0) != 0).astype(float)
    return out


# Filters -------------------------------------------------------------------
lined = pp[COL["lined_up"]].astype("string").str.strip().str.upper()
# normalize common separators
lined = lined.str.replace("-", "_", regex=False).str.replace(" ", "_", regex=False)
mask_pos = lined.isin(LINED_UP_KEEP)

if COL["hadDropback"] is not None:
    drop = to_numeric_series(pp[COL["hadDropback"]])
    mask_pass = drop.fillna(0) != 0
    pass_rule = f"{COL['hadDropback']} != 0"
else:
    # Proxy: row looks like a pass-rush charting row if any rush timing / pressure fields present
    proxy_cols = [COL[k] for k in ("time_to_pressure", "get_off", "causedPressure", "wasInitialPassRusher") if COL[k]]
    mask_pass = False
    for c in proxy_cols:
        mask_pass = mask_pass | pp[c].notna()
    if COL["wasInitialPassRusher"]:
        mask_pass = mask_pass | (to_numeric_series(pp[COL["wasInitialPassRusher"]]).fillna(0) != 0)
    pass_rule = "proxy: non-null pass-rush fields / wasInitialPassRusher"

filt = pp.loc[mask_pos & mask_pass].copy()
print("filter lined_up in", sorted(LINED_UP_KEEP), "& pass rule:", pass_rule)
print("rows in -> kept:", len(pp), "->", len(filt), "(counts only; no row dump)")

# Join season / names
if games is not None:
    gid = resolve_column(games.columns, "gameId", "game_id", required=True)
    filt = filt.merge(games, how="left", left_on=COL["gameId"], right_on=gid)
    if GAMES_TYPE and GAMES_TYPE in filt.columns:
        st = filt[GAMES_TYPE].astype("string").str.upper()
        # Prefer REG if present; keep all if column empty/unexpected
        if st.isin(["REG", "REGULAR", "REGULAR_SEASON"]).any():
            before = len(filt)
            filt = filt.loc[st.isin(["REG", "REGULAR", "REGULAR_SEASON"])].copy()
            print(f"REG filter via {GAMES_TYPE}: {before} -> {len(filt)}")

if players is not None:
    pid = resolve_column(players.columns, "nflId", "nfl_id", required=True)
    filt = filt.merge(players, how="left", left_on=COL["nflId"], right_on=pid, suffixes=("", "_player"))


In [ ]:
# Feature engineering on filtered play rows
work = filt.copy()
idx = work.index


def col_or_nan(colname: str | None) -> pd.Series:
    if colname is None:
        return pd.Series(np.nan, index=idx, dtype="float64")
    return to_numeric_series(work[colname])


# Times (leave NaN; group means skip them)
work["time_to_pressure"] = col_or_nan(COL["time_to_pressure"])
work["time_to_qb_hurry"] = col_or_nan(COL["time_to_qb_hurry"])
work["player_get_off"] = col_or_nan(COL["get_off"])

# Events / binaries
if COL["causedPressure"]:
    work["pressure_event"] = binary_event(work[COL["causedPressure"]])
else:
    work["pressure_event"] = work["time_to_pressure"].notna().astype(float)

# hurry: prefer explicit hurry-time presence as event; else qb hit
if COL["time_to_qb_hurry"]:
    work["hurry_event"] = work["time_to_qb_hurry"].notna().astype(float)
elif COL["qb_hit"]:
    work["hurry_event"] = binary_event(work[COL["qb_hit"]])
else:
    work["hurry_event"] = pd.Series(0.0, index=idx)

if COL["sack_flag"]:
    work["sack_event"] = binary_event(work[COL["sack_flag"]])
else:
    sy = col_or_nan(COL["sack_yards"]).fillna(0)
    hy = col_or_nan(COL["half_sack_yards"]).fillna(0)
    work["sack_event"] = ((sy != 0) | (hy != 0)).astype(float)

work["blitzing"] = binary_event(work[COL["blitzing"]]) if COL["blitzing"] else pd.Series(np.nan, index=idx)
work["unblocked_pressure"] = (
    binary_event(work[COL["unblocked_pressure"]]) if COL["unblocked_pressure"] else pd.Series(np.nan, index=idx)
)
work["quick_pressure"] = (
    binary_event(work[COL["quick_pressure"]]) if COL["quick_pressure"] else pd.Series(np.nan, index=idx)
)

if COL["wasInitialPassRusher"]:
    work["was_initial_pass_rusher"] = binary_event(work[COL["wasInitialPassRusher"]])
else:
    work["was_initial_pass_rusher"] = pd.Series(np.nan, index=idx)

work["lined_up_position"] = lined.loc[idx].astype("string")

print("engineered columns ready; null rates / means only (no row dump):")
for c in [
    "time_to_pressure",
    "time_to_qb_hurry",
    "player_get_off",
    "pressure_event",
    "hurry_event",
    "sack_event",
    "blitzing",
    "unblocked_pressure",
    "quick_pressure",
]:
    s = work[c]
    print(f"  {c:22s} non-null={int(s.notna().sum())}  mean={float(s.mean(skipna=True)) if s.notna().any() else float('nan')}")


In [ ]:
def json_nonnull_list(series: pd.Series) -> str:
    vals = [float(x) for x in series.dropna().tolist()]
    return json.dumps(vals, separators=(",", ":"))


def aggregate(df: pd.DataFrame, group_keys: list[str]) -> pd.DataFrame:
    rows = []
    grouped = df.groupby(group_keys, dropna=False, sort=False)
    for key, g in grouped:
        if not isinstance(key, tuple):
            key = (key,)
        rec = dict(zip(group_keys, key))
        rec["n_plays"] = int(len(g))

        # Times: mean over non-NaN; counts of available times; optional lists
        for time_col, mean_name, n_name, list_name in [
            ("time_to_pressure", "mean_time_to_pressure", "n_time_to_pressure", "times_to_pressure_json"),
            ("time_to_qb_hurry", "mean_time_to_qb_hurry", "n_time_to_qb_hurry", "times_to_qb_hurry_json"),
            ("player_get_off", "mean_player_get_off", "n_player_get_off", "player_get_off_times_json"),
        ]:
            s = g[time_col]
            rec[mean_name] = float(s.mean(skipna=True)) if s.notna().any() else np.nan
            rec[n_name] = int(s.notna().sum())
            if STORE_TIME_LISTS:
                rec[list_name] = json_nonnull_list(s)

        # Event rates (mean of 0/1) and counts (sum)
        for ev_col, rate_name, count_name in [
            ("pressure_event", "pressure_rate", "n_pressures"),
            ("hurry_event", "hurry_rate", "n_hurries"),
            ("sack_event", "sack_rate", "n_sacks"),
        ]:
            s = g[ev_col].fillna(0)
            rec[rate_name] = float(s.mean())
            rec[count_name] = int(s.sum())

        # Optional binary rates
        for opt_col, rate_name, count_name in [
            ("blitzing", "blitz_rate", "n_blitzes"),
            ("unblocked_pressure", "unblocked_pressure_rate", "n_unblocked_pressures"),
            ("quick_pressure", "quick_pressure_rate", "n_quick_pressures"),
            ("was_initial_pass_rusher", "initial_pass_rusher_rate", "n_initial_pass_rusher"),
        ]:
            s = g[opt_col]
            if s.notna().any():
                filled = s.fillna(0)
                rec[rate_name] = float(filled.mean())
                rec[count_name] = int(filled.sum())
            else:
                rec[rate_name] = np.nan
                rec[count_name] = 0

        # Mode lined-up position within group (string counts only)
        if "lined_up_position" in g.columns and g["lined_up_position"].notna().any():
            rec["primary_lined_up_position"] = g["lined_up_position"].mode(dropna=True).iloc[0]
        else:
            rec["primary_lined_up_position"] = pd.NA

        # Carry team / name if constant-ish
        if COL["teamAbbr"] and COL["teamAbbr"] in g.columns:
            rec["teamAbbr"] = g[COL["teamAbbr"]].mode(dropna=True).iloc[0] if g[COL["teamAbbr"]].notna().any() else pd.NA
        if PLAYERS_NAME and PLAYERS_NAME in g.columns:
            rec["displayName"] = g[PLAYERS_NAME].mode(dropna=True).iloc[0] if g[PLAYERS_NAME].notna().any() else pd.NA
        if PLAYERS_POS and PLAYERS_POS in g.columns:
            rec["roster_position"] = g[PLAYERS_POS].mode(dropna=True).iloc[0] if g[PLAYERS_POS].notna().any() else pd.NA

        rows.append(rec)
    return pd.DataFrame(rows)


game_keys = [COL["gameId"], COL["nflId"]]
if GAMES_SEASON and GAMES_SEASON in work.columns:
    # include season on game table for convenience
    game_keys = [GAMES_SEASON, COL["gameId"], COL["nflId"]]

by_game = aggregate(work, game_keys)
print("by_game shape:", by_game.shape)

if GAMES_SEASON and GAMES_SEASON in work.columns:
    by_season = aggregate(work, [GAMES_SEASON, COL["nflId"]])
else:
    raise RuntimeError("Cannot build season aggregates without games.season — join games.csv first.")
print("by_season shape:", by_season.shape)

# Preview headers + tiny describe only (no raw competition rows)
print("by_game columns:", list(by_game.columns))
print("by_season columns:", list(by_season.columns))
print(by_game[["n_plays", "pressure_rate", "mean_time_to_pressure", "mean_player_get_off"]].describe())


In [ ]:
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
by_game.to_csv(OUT_GAME, index=False)
by_season.to_csv(OUT_SEASON, index=False)
print("Wrote:", OUT_GAME)
print("Wrote:", OUT_SEASON)
print("Done. Keep DerivedData/ out of git and out of agent-readable paths.")


## Output column summary

**Keys:** `season` (season table; also on game table when joined), `gameId` (game table), `nflId`; optional `displayName`, `roster_position`, `teamAbbr`, `primary_lined_up_position`.

**Volume:** `n_plays`.

**Times (mean over non-NaN + availability count + optional JSON lists):**
`mean_time_to_pressure`, `n_time_to_pressure`, `times_to_pressure_json`,
`mean_time_to_qb_hurry`, `n_time_to_qb_hurry`, `times_to_qb_hurry_json`,
`mean_player_get_off`, `n_player_get_off`, `player_get_off_times_json`.

**Events (rate = mean of 0/1; count = sum):**
`pressure_rate` / `n_pressures`, `hurry_rate` / `n_hurries`, `sack_rate` / `n_sacks`,
optional `blitz_rate` / `n_blitzes`, `unblocked_pressure_rate` / `n_unblocked_pressures`,
`quick_pressure_rate` / `n_quick_pressures`, `initial_pass_rusher_rate` / `n_initial_pass_rusher`.
